In [2]:
import fastf1
import pandas as pd
import numpy as np
import os
import warnings
import logging
from tqdm import tqdm
warnings.filterwarnings('ignore')
os.chdir(r'C:\Users\adity\Desktop\BoxBox')
print(os.getcwd())

C:\Users\adity\Desktop\BoxBox


SETUP

Create all required folders if they don't exist yet
loggiing gives us a proper log file instead of just print statements

This way if Phase 1 fails partway through, you can read data/raw/collection_log.txt to see exactly what went wrong

In [3]:
logger_name = __name__  
log = logging.getLogger(logger_name)
if log.hasHandlers():
    log.handlers.clear()

log.setLevel(logging.INFO)

formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')

file_handler = logging.FileHandler(
    r'C:\Users\adity\Desktop\BoxBox\data\outputs\phase1_log.txt'
)
file_handler.setFormatter(formatter)

stream_handler = logging.StreamHandler()
stream_handler.setFormatter(formatter)

log.addHandler(file_handler)
log.addHandler(stream_handler)

#Enable fastf1 cache so data is stored locally after first download
fastf1.Cache.enable_cache(r'C:\Users\adity\Desktop\BoxBox\cache')


2024 RACE CALENDER 

Each entry is a dict with the event name exactly as FastF1 expects it, and a flag for whether it's a sprint weekend

Sprint weekends use a different session format, so we need to know which ones they are before loading

In [4]:
CALENDAR_2024 = [
    {'name': 'Bahrain', 'round': 1, 'sprint':False},
    {'name': 'Saudi Arabia', 'round': 2, 'sprint':False},
    {'name': 'Australia', 'round': 3, 'sprint':False},
    {'name': 'Japan', 'round': 4, 'sprint':False},
    {'name': 'China', 'round': 5, 'sprint':True},
    {'name': 'Miami', 'round': 6, 'sprint':True},
    {'name': 'Emilia Romagna', 'round': 7, 'sprint':False},
    {'name': 'Monaco', 'round': 8, 'sprint':False},
    {'name': 'Canada', 'round': 9, 'sprint': False},
    {'name': 'Spain', 'round': 10, 'sprint': False},
    {'name': 'Austria', 'round': 11, 'sprint': True},
    {'name': 'Great Britain', 'round': 12, 'sprint': False},
    {'name': 'Hungary', 'round': 13, 'sprint': False},
    {'name': 'Belgium', 'round': 14, 'sprint': False},
    {'name': 'Netherlands', 'round': 15, 'sprint': False},
    {'name': 'Italy', 'round': 16, 'sprint': False},
    {'name': 'Azerbaijan', 'round': 17, 'sprint': False},
    {'name': 'Singapore', 'round': 18, 'sprint': False},
    {'name': 'United States', 'round': 19, 'sprint': True},
    {'name': 'Mexico', 'round': 20, 'sprint': False},
    {'name': 'Brazil', 'round': 21, 'sprint': True},
    {'name': 'Las Vegas', 'round': 22, 'sprint': False},
    {'name': 'Qatar', 'round': 23, 'sprint': True},
    {'name': 'Abu Dhabi', 'round': 24, 'sprint': False},
]

Session Loader

This is the single function responsible for loading any FastF1 session. It's separated out so all 5 collection functions can resue it without duplicating try/except logic

Returns the loaded session object, or None if loading fails

In [5]:
def load_session(round_number, session_type, year=2024):
    try:
        session = fastf1.get_session(year, round_number, session_type)
        session.load(
            laps=True, #We want the lap data
            telemetry=False, #skipping the raw sensor data here (will collect them separately afterwards)
            weather=True, #sir, temp, track temp, humidity per lamp
            messages= False #Skip the radio transcripts
        )
        return session
    except Exception as e:
        log.error(f"Failed to load session R{round_number} {session_type}: {e}")
        return None

COLLECT LAP DATA

Pulls every lap from every driver from every race. 

We keep almost all columns here - in phase 2 we will decide what to drop. The only trnsformation we do is converting LapTime from timedelta to seconds, because timedelta objects can't be stored in a CSV cleanly

In [6]:
def collect_lap_data():
    log.info("COLLECTS LAP DATA")
    log.info("=" * 50)

    all_laps =[]
    failed_races = []

    for race in tqdm(CALENDAR_2024, desc="Lap data"):
        session = load_session(race['round'], 'R')

        if session is None:
            failed_races.append(race['name'])
            continue

        laps = session.laps.copy()

        '''Tag each row with identifiers so we know which
        race it came from after concatenation'''
        laps['CircuitName'] = race['name']
        laps['RoundNumber'] = race['round']
        laps['Year'] = 2024
        laps['IsSprintWeekend'] = race['sprint']

        '''Converting LapTime timedelta -> seconds float
        dt.total_seconds() handles NaT gracefully -> NaN'''
        laps['LapTimeSeconds'] = laps['LapTime'].dt.total_seconds()

        # Convert sector times from timedelta -> seconds as well
        for sector in ['Sector1Time', 'Sector2Time', 'Sector3Time']:
            if sector in laps.columns:
                laps[f'{sector}Seconds'] = laps[sector].dt.total_seconds()

        '''Convert pit times to boolean flags - we don't need
        the exact timestamps here, just whether a pit happened'''
        laps['IsPitInLap'] = laps['PitInTime'].notna()
        laps['IsPitOutLap'] = laps['PitOutTime'].notna()

        '''Drop the raw timedelta columns that don't 
        serialize cleanly to CSV'''
        cols_to_drop = [
            'LapTime', 'PitInTime', 'PitOutTime',
            'Sector1Time', 'Sector2Time', 'Sector3Time',
            'Sector1SessionTime', 'Sector2SessionTime',
            'Sector3SessionTime', 'LapStartTime', 'Time'
        ]
        laps = laps.drop(
            columns = [c for c in cols_to_drop if c in laps.columns]
        )

        log.info(f" {race['name']}: {len(laps)} laps collected")
        all_laps.append(laps)
    
    if not all_laps:
        log.error("No lap data collected at all. Check internet/FastF1")
        return
    
    #Stack all races into one DataFrame
    full_laps = pd.concat(all_laps, ignore_index=True)

    output_path = r'C:\Users\adity\Desktop\BoxBox\data\raw\raw_laps.csv'
    full_laps.to_csv(output_path, index=False)

    log.info(f"\nLap data saved: {output_path}")
    log.info(f"Total rows: {len(full_laps)}")
    log.info(f"Circuits collected: {full_laps['CircuitName'].nunique()}")
    if failed_races:
        log.warning(f"Failed race: {failed_races}")

COLLECT PIT STOP DATA

FastF1 provides pit stop data directly via session.laps - we filter for laps where the driver pitted and extract the relevant timing information

This tells us: who pitted on which lap and how long the stationary stop was, and what compound they went onto

In [7]:
def collect_pit_stop_data():
    log.info("=" * 50)
    log.info("COLLECTING PIT STOP DATA")
    log.info("=" * 50)

    all_pitstops = []
    failed_races = []

    for race in tqdm(CALENDAR_2024, desc="Pit stop data"):
        session = load_session(race['round'], 'R')

        if session is None:
            failed_races.append(race['name'])
            continue

        laps = session.laps.copy()

        # A pit stop lap is one where PitInTime is not null
        pit_laps = laps[laps['PitInTime'].notna()].copy()

        if pit_laps.empty:
            log.warning(f"  {race['name']}: No pit stop data found")
            continue

        # Build a clean pit stop table
        pit_data = pd.DataFrame()
        pit_data['Driver'] = pit_laps['Driver']
        pit_data['Team'] = pit_laps['Team']
        pit_data['LapNumber'] = pit_laps['LapNumber']
        pit_data['Stint'] = pit_laps['Stint']
        pit_data['CompoundOff'] = pit_laps['Compound']

        # Sort so shift(-1) correctly grabs each driver's NEXT lap
        laps_sorted = laps.sort_values(['Driver', 'LapNumber'])

        # Compound going ON — lives on the NEXT lap
        next_compound = laps_sorted.groupby('Driver')['Compound'].shift(-1)
        pit_data['CompoundOn'] = next_compound.loc[pit_laps.index].values

        # PitOutTime ALSO lives on the NEXT lap, not the pit-in lap —
        # this is the actual fix. Pulling it from the same row as
        # PitInTime (the old bug) returned NaN or nonsensical values.
        next_pit_out = laps_sorted.groupby('Driver')['PitOutTime'].shift(-1)
        actual_pit_out = next_pit_out.loc[pit_laps.index]

        # Calculate duration: NEXT lap's PitOutTime minus
        # CURRENT lap's PitInTime
        pit_in = pit_laps['PitInTime']

        pit_duration = (
            pd.to_timedelta(actual_pit_out.values) -
            pd.to_timedelta(pit_in.values)
        )
        pit_data['PitDurationSeconds'] = pd.Series(pit_duration).dt.total_seconds().values

        pit_data['CircuitName'] = race['name']
        pit_data['RoundNumber'] = race['round']
        pit_data['Year'] = 2024

        # Drop rows where we still couldn't calculate a valid duration
        # (e.g. driver pitted on the very last lap, so there's no "next lap")
        valid_count = pit_data['PitDurationSeconds'].notna().sum()
        log.info(f"  {race['name']}: {len(pit_data)} pit stops "
                 f"({valid_count} with valid duration)")

        all_pitstops.append(pit_data)

    if not all_pitstops:
        log.error("No pit stop data collected.")
        return

    full_pitstops = pd.concat(all_pitstops, ignore_index=True)

    output_path = r'C:\Users\adity\Desktop\BoxBox\data\raw\pit_stops.csv'
    full_pitstops.to_csv(output_path, index=False)

    log.info(f"\nPit stop data saved: {output_path}")
    log.info(f"Total pit stops: {len(full_pitstops)}")
    log.info(f"Valid durations: {full_pitstops['PitDurationSeconds'].notna().sum()}")
    log.info(f"Duration stats:\n{full_pitstops['PitDurationSeconds'].describe()}")
    if failed_races:
        log.warning(f"Failed races: {failed_races}")

COLLECT RACE RESULTS

session.results gives us the dinal classification for each race. We need grid position (where they started) and final position (where they finished) to train the Strategy Outcome model. 

We also collect points scored and status (finished/DNF/DSQ)

In [8]:
def collect_race_results():
    log.info("COLLECTING RACE RESULTS")
    log.info("=" * 50)

    all_results = []
    failed_races = []

    for race in tqdm(CALENDAR_2024, desc='Race results'):
        session = load_session(race['round'], 'R')

        if session is None:
            failed_races.append(race['name'])
            continue
        
        results = session.results.copy()

        if results is None or results.empty:
            log.warning(f"{race['name']}: No results data found")
            continue

        '''Select only thee columns we need
        Not all columns are always present so we check first'''
        columns_wanted = [
            'DriverNumber', 'Abbreviation', 'FullName',
            'TeamName', 'GridPosition', 'Position',
            'Points', 'Status', 'ClassifiedPosition'
        ]

        cols = [c for c in columns_wanted if c in results.columns]
        result_df = results[cols].copy()

        result_df['CircuitName'] = race['name']
        result_df['RoundNumber'] = race['round']
        result_df['Year'] = 2024

        '''Position gain/loss - positive means moved forward
        This becomes the label for our Strategy Outcome model'''
        if 'GridPosition' in result_df.columns and 'Position' in result_df.columns:
            result_df['PositionChange'] = (
                pd.to_numeric(result_df['GridPosition'], errors='coerce') -
                pd.to_numeric(result_df['Position'], errors='coerce')
            )
        
        log.info(f" {race['name']}: {len(result_df)} driver results collected")
        all_results.append(result_df)

    if not all_results:
        log.error("No race results collected")
        return
    
    full_results = pd.concat(all_results, ignore_index=True)

    output_path = r'C:\Users\adity\Desktop\BoxBox\data\raw\race_results.csv'
    full_results.to_csv(output_path, index=False)

    log.info(f"\nRace results saved: {output_path}")
    log.info(f"Total rows: {len(full_results)}")
    if failed_races:
        log.warning(f"Failed races: {failed_races}")

COLLECT WEATHER DATA

session.weather_data gives timestamped weather readings throughout the race - air temp, track temp, humidity, wind speed, wind direction, and rainfall flag.

Weather is sampled every few minutes, not every lap, so in Next phase we have to merge it into the lap data by maching the closest timestamp

In [9]:
def collect_weather_data():
    log.info("COLLECTING WEATHER DATA")
    log.info("=" * 50)

    all_weather = []
    failed_races =[]

    for race in tqdm(CALENDAR_2024, desc='Weather data'):
        session = load_session(race['round'], 'R')

        if session is None:
            failed_races.append(race['name'])
            continue

        weather = session.weather_data

        if weather is None or weather.empty:
            log.warning(f" {race['name']}: No weather data found")
            continue

        weather = weather.copy()

        '''Convert the session timestamp to seconds from race start
        so it's mergeable with lap data in next phase'''
        if 'Time' in weather.columns:
            weather['TimeSeconds'] = weather['Time'].dt.total_seconds()
            weather = weather.drop(columns=['Time'])

        weather['CircuitName'] = race['name']
        weather['RoundNumber'] = race['round']
        weather['Year'] = 2024

        log.info(f" {race['name']}: {len(weather)} weather readings collected")
        all_weather.append(weather)

    if not all_weather:
        log.error("No weather data collected")
        return
    
    full_weather = pd.concat(all_weather, ignore_index=True)
    output_path = r'C:\Users\adity\Desktop\BoxBox\data\raw\weather.csv'

    log.info(f"\nWeather data saved: {output_path}")
    log.info(f"Total Rows: {len(full_weather)}")
    if failed_races:
        log.warning(f"Failed races: {failed_races}")

COLLECT TELEMETRY FOR CIRCUIT LAPS 

For each circuit we load one lap of telementry from one driver - we only need the X/Y GPS coordinates to draw the track outline. We pick the fastest qualifier's fastest lap from qualifying (not the race) since that gives the cleanest single lap representation of the circuit.

We load qualifying ('Q') separately for this purpose 


In [10]:
def collect_telemetry():
    log.info("COLLECT TELEMETRY FOR CIRCUIT MAPS")
    log.info("=" * 50)

    all_telemetry = []
    failed_races = []

    for race in tqdm(CALENDAR_2024, desc="Telemetry"):
        try:
            #Load qualifying session for the clean lap shape
            session = fastf1.get_session(2024, race['round'], 'Q')

            '''Telemetry here is true because we specifically need
            the GPS coordinates streams'''
            session.load(
                laps=True,
                telemetry=True,
                weather=False,
                messages=False
            )
            #Get the fastest lap of the whole session
            fastest_lap = session.laps.pick_fastest()

            if fastest_lap is None:
                log.warning(f" {race['name']}: No feature lap found in the qualifying round")
                failed_races.append(race['name'])
                continue

            '''Get the telemetry for that lap
            position_data gives X,Y,Z coordinates from GPS'''
            telemetry = fastest_lap.get_pos_data()

            if telemetry is None or telemetry.empty:
                log.warning(f" {race['name']}: No telemetry data found")
                continue

            '''We only need the X and Y coordinates for 2D map
            Z s the altitude which we won't be using for the flat map
            It's too noisy and can break the 2D framing of the track'''

            cols_to_keep = [c for c in ['X', 'Y', 'Z', 'Status']
                            if c in telemetry.columns]
            telemetry = telemetry[cols_to_keep]

            telemetry['CircuitName'] = race['name']
            telemetry['RoundNumber'] = race['round']

            log.info(f" {race['name']}: {len(telemetry)} telemetry points collected")
            all_telemetry.append(telemetry)
        
        except Exception as e:
            log.error(f" {race['name']}: Telemtry failed - {e}")
            failed_races.append(race['name'])
            continue
    
    if not all_telemetry:
        log.error("No telemetry data collected")
        return
    
    full_telemetry = pd.concat(all_telemetry, ignore_index=True)

    output_path = r'C:\Users\adity\Desktop\BoxBox\data\raw\telemetry.csv'
    full_telemetry.to_csv(output_path, index=False)

    log.info(f"\nTelemetry saved: {output_path}")
    log.info(f"Circuits with maps: {full_telemetry['CircuitName'].nunique()}")
    if failed_races:
        log.error(f"Failed Circuits: {failed_races}")


MAIN PIPELINE

Runs all 5 collectors in sequence. Each one is independent, so if any
one fails entirely, the others still run and save their own files. 

At the end, we will acquire 5 csv's and print print a summary of what we saved to have an intel of what we have

In [11]:
def main():
    log.info("BOXBOX PHASE 1: Data collection")
    log.info("2024 F1 Season | 24 Races")
    log.info("-" * 50)

    collect_lap_data()
    collect_pit_stop_data()
    collect_race_results()
    collect_weather_data()
    collect_telemetry()

    log.info("\n" + "-" * 50)
    log.info("Summary")
    log.info("-" * 50)

    files = {
    'Lap Data':     r'C:\Users\adity\Desktop\BoxBox\data\raw\raw_laps.csv',
    'Pit Stops':    r'C:\Users\adity\Desktop\BoxBox\data\raw\pit_stops.csv',
    'Race Results': r'C:\Users\adity\Desktop\BoxBox\data\raw\race_results.csv',
    'Weather':      r'C:\Users\adity\Desktop\BoxBox\data\raw\weather.csv',
    'Telemetry':    r'C:\Users\adity\Desktop\BoxBox\data\raw\telemetry.csv',
    }
    for name, path in files.items():
        if os.path.exists(path):
            size_mb = os.path.getsize(path) / (1024 * 1024)
            df = pd.read_csv(path, nrows=1)
            full_df = pd.read_csv(path)
            log.info(f"{name}: {len(full_df)} rows | {size_mb:.1f} MB -> {path}")
        else:
            log.warning(f" {name}: NOT SAVED- check log for errors")


if __name__ == '__main__':
    main()

2026-07-28 07:10:07,923 - INFO - BOXBOX PHASE 1: Data collection
2026-07-28 07:10:07,923 - INFO - 2024 F1 Season | 24 Races
2026-07-28 07:10:07,924 - INFO - --------------------------------------------------
2026-07-28 07:10:07,925 - INFO - COLLECTS LAP DATA
2026-07-28 07:10:07,926 - INFO - ==================================================
Lap data:   0%|          | 0/24 [00:00<?, ?it/s]core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for weather_data
core           INFO 	